In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
    confusion_matrix,
    precision_recall_curve,
    average_precision_score,
    roc_curve,
    roc_auc_score
)

# ==========================================
# 1. Load Real Imbalanced Dataset
# ==========================================

dataset = fetch_ucirepo(id=468)

X = dataset.data.features.copy()
y = dataset.data.targets["Revenue"].astype(int)

print("Dataset shape:", X.shape)
print("\nClass distribution:")
print(y.value_counts().sort_index())

print("\nClass percentages:")
print(y.value_counts(normalize=True).sort_index() * 100)


# ==========================================
# 2. Train/Test Split
# ==========================================

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print("\nTrain size:", len(X_train))
print("Test size:", len(X_test))


# ==========================================
# 3. Data Preprocessing
# ==========================================

categorical_cols = [
    "Month",
    "OperatingSystems",
    "Browser",
    "Region",
    "TrafficType",
    "VisitorType",
    "Weekend"
]

numeric_cols = [
    col for col in X.columns
    if col not in categorical_cols
]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"),
         categorical_cols)
    ]
)


# ==========================================
# 4. Logistic Regression Model
# ==========================================

model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=2000,
        random_state=42
    ))
])

model.fit(X_train, y_train)

# Probability of positive class (Revenue = 1)
probs = model.predict_proba(X_test)[:, 1]


# ==========================================
# 5. Compare Different Thresholds
# ==========================================

results = []

for threshold in [0.5, 0.3, 0.1]:

    predictions = (probs >= threshold).astype(int)

    results.append({
        "Threshold": threshold,
        "Precision": precision_score(
            y_test, predictions, zero_division=0),
        "Recall": recall_score(
            y_test, predictions, zero_division=0),
        "F1": f1_score(
            y_test, predictions, zero_division=0),
        "Accuracy": accuracy_score(
            y_test, predictions)
    })

results_df = pd.DataFrame(results)

print("\nThreshold Comparison:")
print(results_df.round(3))


# ==========================================
# 6. Precision-Recall Curve
# ==========================================

precision, recall, pr_thresholds = precision_recall_curve(
    y_test, probs
)

ap = average_precision_score(y_test, probs)

positive_rate = y_test.mean()

plt.figure(figsize=(8, 5))

plt.plot(
    recall, precision,
    label=f"Average Precision = {ap:.3f}"
)

plt.axhline(
    y=positive_rate,
    linestyle="--",
    label=f"Positive Rate = {positive_rate:.3f}"
)

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve - Real Dataset")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


# ==========================================
# 7. ROC Curve
# ==========================================

fpr, tpr, roc_thresholds = roc_curve(
    y_test, probs
)

roc_auc = roc_auc_score(y_test, probs)

plt.figure(figsize=(8, 5))

plt.plot(
    fpr, tpr,
    label=f"ROC-AUC = {roc_auc:.3f}"
)

plt.plot(
    [0, 1], [0, 1],
    linestyle="--",
    label="Random Classifier"
)

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - Real Dataset")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


# ==========================================
# 8. Final Evaluation
# ==========================================

print(f"Average Precision: {ap:.3f}")
print(f"ROC-AUC: {roc_auc:.3f}")

predictions = (probs >= 0.5).astype(int)

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, predictions))
